# LangGraph Basics — A Progressive, Hands-On Tour

This notebook builds up LangGraph concept by concept. Each step adds one
concept using a progressively familiar graph pattern; most examples are
deliberately standalone so you can compare one idea in isolation.

Every step below has: a theory explanation (what/why/how, plus related
concepts), a Mermaid diagram of that step's graph topology, runnable code
against a real LLM (Anthropic or OpenAI — your choice, see setup below), and
visible output proving the concept actually works.

**Provider note:** this repo has no mock mode. Every LLM call below is real.

## Learning path

1. Build fixed, branching, and cyclic graphs.
2. Add persistence, tools, streaming, approval, composition, and parallelism.
3. Use the exercises to modify the graph rather than only reading its output.

### How to study each section

Before running the code, predict the next node and the state update. After
running it, inspect the output and complete the exercise. The assertions in
the code are intentionally small: they model the tests students should write
for graph routing, termination, reducers, and tool behavior.

## Setup

Set the `PROVIDER` flag below to `"anthropic"` or `"openai"` — this single
flag controls which API the rest of the notebook uses. There is no silent
auto-detection: if the matching key isn't in `.env`, `get_llm()` fails
loudly rather than falling back to the other provider.

In [ ]:
import os
import time
import operator
from typing import TypedDict, Annotated, Literal

from dotenv import load_dotenv

load_dotenv()

# --- THE FLAG: change this to switch providers for the whole notebook ---
PROVIDER = "anthropic"  # or "openai"

ANTHROPIC_MODEL = "claude-sonnet-5"
OPENAI_MODEL = "gpt-4o"


def get_llm(model: str | None = None):
    """Returns a LangChain chat model chosen by the PROVIDER flag above.
    Fails loudly if the matching key is missing -- no fallback to the other
    provider, by design (see teaching_brief.md constraints)."""
    if PROVIDER == "anthropic":
        key = os.environ.get("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("PROVIDER='anthropic' but ANTHROPIC_API_KEY is not set in .env")
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=model or ANTHROPIC_MODEL, api_key=key)
    elif PROVIDER == "openai":
        key = os.environ.get("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("PROVIDER='openai' but OPENAI_API_KEY is not set in .env")
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model or OPENAI_MODEL, api_key=key)
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER!r} -- use 'anthropic' or 'openai'")


def get_text(response) -> str:
    """Extracts plain text from a chat model response. Anthropic sometimes
    returns `.content` as a list of content blocks rather than a plain
    string (e.g. when a thinking/text block split occurs) -- this
    normalizes both providers to a single string so downstream code
    (splitting into lines, etc.) never breaks on a list."""
    content = response.content
    if isinstance(content, str):
        return content
    return "".join(block.get("text", "") for block in content if isinstance(block, dict))


llm = get_llm()
print(f"Using provider: {PROVIDER}  |  model: {llm.model if hasattr(llm, 'model') else llm}")
print(get_text(llm.invoke("Reply with exactly one word: ready")))

## Step (a) — Basic StateGraph: State, Node, Normal Edges, START, END

### Theory
LangGraph models an application as a **graph**: a `State` (shared data
passed between steps), **nodes** (plain Python functions that read state and
return a partial update), and **edges** (which node runs next). `START` and
`END` are special sentinel nodes marking entry and exit.

- **State** is usually a `TypedDict` (or Pydantic model) — a schema, not an
  instance. LangGraph merges each node's returned dict into the running
  state.
- A **node** takes the current state and returns a dict of the fields it
  wants to update (not the whole state — LangGraph merges for you, unless a
  field uses a custom reducer, which we'll meet in step (l)).
- A **normal edge** (`add_edge(a, b)`) is a fixed, unconditional hop — always
  go from `a` straight to `b`.
- `START` and `END` aren't nodes you write logic for — they just mark where
  execution begins and terminates.

**Related concepts (don't confuse yet):** a normal edge is *not* the same as
a conditional edge (step b, which picks the next node dynamically) or a
tool-routed edge (steps e/f, where the LLM's tool call decides the path).
This step is the simplest possible case: one fixed path, no branching.

### Implementation
One `State` TypedDict with a single `text` field, one node that appends to
it, wired `START -> greet -> END`.

In [ ]:
class BasicState(TypedDict):
    text: str


def greet_node(state: BasicState) -> BasicState:
    return {"text": state["text"] + " -> greeted"}


from langgraph.graph import StateGraph, START, END

basic_builder = StateGraph(BasicState)
basic_builder.add_node("greet", greet_node)
basic_builder.add_edge(START, "greet")
basic_builder.add_edge("greet", END)

basic_app = basic_builder.compile()

result = basic_app.invoke({"text": "hello"})
print("Result:", result)
print()
print("Mermaid source (from LangGraph itself):")
print(basic_app.get_graph().draw_mermaid())

### Mermaid diagram — Step (a) topology

```mermaid
graph TD;
    START([START]) --> greet[greet];
    greet --> END([END]);
```

## Step (a variant) — Same Topology, Node Is an LLM Call

### Theory
Everything about the graph shape from step (a) is identical -- one node,
`START -> node -> END`. The only thing that changes is what the node's
*body* does: instead of a plain Python string operation, the node calls the
LLM and returns its response. This is the core idea that makes LangGraph
useful for agents at all -- a node is just "a function that returns a state
update," and that function is free to be a deterministic transform (step a)
or a model call (here). LangGraph itself doesn't know or care which.

**Related concept:** don't confuse "a node that calls an LLM" with "an
agent" -- an agent (as commonly meant) is usually a node (or small
subgraph) that calls an LLM *and* can decide to call tools and loop, which
we build up starting at step (e). A single LLM-call node is the simplest
building block that pattern is made of.

### Implementation
Same `START -> llm_greet -> END` shape as step (a), but `llm_greet` asks the
LLM to rewrite the input as a greeting instead of string-concatenating.

In [ ]:
class LLMBasicState(TypedDict):
    text: str


def llm_greet_node(state: LLMBasicState) -> LLMBasicState:
    response = get_text(
        get_llm().invoke(f"Rewrite this as one warm, natural sentence: {state['text']}")
    )
    return {"text": response}


llm_basic_builder = StateGraph(LLMBasicState)
llm_basic_builder.add_node("llm_greet", llm_greet_node)
llm_basic_builder.add_edge(START, "llm_greet")
llm_basic_builder.add_edge("llm_greet", END)
llm_basic_app = llm_basic_builder.compile()

result = llm_basic_app.invoke({"text": "new user just signed up"})
print("Result:", result)
print()
print(llm_basic_app.get_graph().draw_mermaid())

### Mermaid diagram — Step (a variant) topology (identical shape to step a)

```mermaid
graph TD;
    START([START]) --> llm_greet["llm_greet (LLM call)"];
    llm_greet --> END([END]);
```

## Step (b) — Conditional Edges

### Theory
A **conditional edge** (`add_conditional_edges`) picks the next node at
runtime by calling a routing function on the current state, instead of
always going to a fixed node. The routing function returns a key, and a
mapping dict tells LangGraph which node that key corresponds to.

This is the mechanism behind every "if/else" in a graph: "is this input
long or short?", "did the answer pass the check?", "should we retry?".

**Related concepts:** conditional edges route based on *state you compute
yourself* (e.g. a length check, a classifier result). Tool-based routing
(step f) looks similar but the routing decision comes from the LLM's own
tool-call output, not a function you write. Don't conflate the two — a
conditional edge is the general mechanism; tool routing is one particular
use of it via `tools_condition`.

### Implementation
Extend step (a)'s graph: after `greet`, a conditional edge checks the text
length and routes to either a `short_reply` or `long_reply` node.

In [ ]:
class BranchState(TypedDict):
    text: str
    path_taken: str


def greet_node_b(state: BranchState) -> BranchState:
    return {"text": state["text"] + " -> greeted"}


def route_on_length(state: BranchState) -> Literal["short_reply", "long_reply"]:
    return "short_reply" if len(state["text"]) < 20 else "long_reply"


def short_reply(state: BranchState) -> BranchState:
    return {"path_taken": "short"}


def long_reply(state: BranchState) -> BranchState:
    return {"path_taken": "long"}


branch_builder = StateGraph(BranchState)
branch_builder.add_node("greet", greet_node_b)
branch_builder.add_node("short_reply", short_reply)
branch_builder.add_node("long_reply", long_reply)
branch_builder.add_edge(START, "greet")
branch_builder.add_conditional_edges(
    "greet", route_on_length, {"short_reply": "short_reply", "long_reply": "long_reply"}
)
branch_builder.add_edge("short_reply", END)
branch_builder.add_edge("long_reply", END)

branch_app = branch_builder.compile()

print(branch_app.invoke({"text": "hi", "path_taken": ""}))
print(branch_app.invoke({"text": "a much longer input string here", "path_taken": ""}))
short_result = branch_app.invoke({"text": "hi", "path_taken": ""})
long_result = branch_app.invoke({"text": "a much longer input string here", "path_taken": ""})
assert short_result["path_taken"] == "short"
assert long_result["path_taken"] == "long"
print("Conditional routing assertions passed")
print()
print(branch_app.get_graph().draw_mermaid())

### Mermaid diagram — Step (b) topology

```mermaid
graph TD;
    START([START]) --> greet[greet];
    greet -- short --> short_reply[short_reply];
    greet -- long --> long_reply[long_reply];
    short_reply --> END([END]);
    long_reply --> END([END]);
```

## Step (b variant) — Same Topology, Routing Decision and Branches Are LLM Calls

### Theory
Step (b)'s routing function (`route_on_length`) was a deterministic
`len()` check -- it always gives the same answer for the same input. Here,
the routing function itself makes an LLM call to classify the input, so
the decision can vary based on the model's judgment of meaning, not just a
measurable property of the text. The branch nodes are also now LLM calls
that generate a real reply, not just a label.

**Related concept:** this "classify, then branch" pattern generalizes to
the multi-tool selection in step (f) -- the difference is that here *we*
write the classification prompt and the routing function explicitly,
whereas tool selection lets the LLM pick from bound tool schemas directly
without us writing a separate classifier.

### Implementation
Same `START -> classify -> (question_reply | statement_reply) -> END`
shape as step (b), but `classify` asks the LLM whether the input is a
question or a statement, and each branch node asks the LLM for a reply.

In [ ]:
class LLMBranchState(TypedDict):
    text: str
    kind: str
    reply: str


def classify_node(state: LLMBranchState) -> LLMBranchState:
    raw = get_text(
        get_llm().invoke(
            f"Classify this input as exactly one word, 'question' or 'statement': {state['text']}"
        )
    ).strip().lower()
    return {"kind": "question" if "question" in raw else "statement"}


def route_on_kind(state: LLMBranchState) -> Literal["question_reply", "statement_reply"]:
    return "question_reply" if state["kind"] == "question" else "statement_reply"


def question_reply(state: LLMBranchState) -> LLMBranchState:
    return {"reply": get_text(get_llm().invoke(f"Answer briefly: {state['text']}"))}


def statement_reply(state: LLMBranchState) -> LLMBranchState:
    return {"reply": get_text(get_llm().invoke(f"Acknowledge this statement in one short sentence: {state['text']}"))}


llm_branch_builder = StateGraph(LLMBranchState)
llm_branch_builder.add_node("classify", classify_node)
llm_branch_builder.add_node("question_reply", question_reply)
llm_branch_builder.add_node("statement_reply", statement_reply)
llm_branch_builder.add_edge(START, "classify")
llm_branch_builder.add_conditional_edges(
    "classify", route_on_kind, {"question_reply": "question_reply", "statement_reply": "statement_reply"}
)
llm_branch_builder.add_edge("question_reply", END)
llm_branch_builder.add_edge("statement_reply", END)
llm_branch_app = llm_branch_builder.compile()

r1 = llm_branch_app.invoke({"text": "What is LangGraph?", "kind": "", "reply": ""})
print("Input: 'What is LangGraph?'  -> kind:", r1["kind"], " reply:", r1["reply"])

r2 = llm_branch_app.invoke({"text": "I just finished building my first agent.", "kind": "", "reply": ""})
print("Input: 'I just finished building my first agent.'  -> kind:", r2["kind"], " reply:", r2["reply"])
print()
print(llm_branch_app.get_graph().draw_mermaid())

### Mermaid diagram — Step (b variant) topology (identical shape to step b)

```mermaid
graph TD;
    START([START]) --> classify["classify (LLM call)"];
    classify -- question --> question_reply["question_reply (LLM call)"];
    classify -- statement --> statement_reply["statement_reply (LLM call)"];
    question_reply --> END([END]);
    statement_reply --> END([END]);
```

## Step (c) — Loop / Cycle

### Theory
Unlike a plain LangChain LCEL chain (which is a DAG — no going back), a
LangGraph graph can **cycle**: a conditional edge can route back to a node
that already ran. This is how "keep refining until good enough" or "retry
until success" loops are built.

A loop needs a **termination condition** in state (e.g. a counter, a quality
flag) — otherwise it runs forever. `add_conditional_edges` checks that
condition each pass and decides "loop again" vs "exit to END".

**Related concept:** this is different from the *recursion limit* (step g),
which is a hard safety cap LangGraph enforces regardless of your own loop
logic — your termination condition is the intended exit; the recursion
limit is the safety net if that logic has a bug.

### Implementation
A node increments a counter each pass; a conditional edge loops back to
itself while `counter < 3`, then exits.

In [ ]:
class LoopState(TypedDict):
    counter: int
    log: Annotated[list[str], operator.add]


def increment(state: LoopState) -> LoopState:
    new_count = state["counter"] + 1
    return {"counter": new_count, "log": [f"pass {new_count}"]}


def should_continue(state: LoopState) -> Literal["increment", "__end__"]:
    return "increment" if state["counter"] < 3 else "__end__"


loop_builder = StateGraph(LoopState)
loop_builder.add_node("increment", increment)
loop_builder.add_edge(START, "increment")
loop_builder.add_conditional_edges("increment", should_continue, {"increment": "increment", "__end__": END})

loop_app = loop_builder.compile()

print(loop_app.invoke({"counter": 0, "log": []}))
loop_result = loop_app.invoke({"counter": 0, "log": []})
assert loop_result["counter"] == 3
assert len(loop_result["log"]) == 3
print("Loop termination assertions passed")
print()
print(loop_app.get_graph().draw_mermaid())

### Mermaid diagram — Step (c) topology

```mermaid
graph TD;
    START([START]) --> increment[increment];
    increment -- counter < 3 --> increment;
    increment -- counter >= 3 --> END([END]);
```

## Step (c variant) — Same Loop Shape, Self-Refinement Driven by an LLM Judge

### Theory
Step (c)'s loop condition was a plain counter (`counter < 3`). Here, the
loop is a genuine **self-refinement** pattern: a `write_draft` node asks the
LLM to produce something, a `judge_draft` node asks a *second* LLM call to
critique it, and the conditional edge loops back to `write_draft` (carrying
the judge's feedback into the next attempt) until the judge approves or a
max-attempts safety cap is hit. This is the same shape as "corrective RAG"
mentioned earlier in this conversation -- generate, grade, and loop only if
grading fails.

**Related concept:** the max-attempts cap here is *your own* termination
logic, distinct from the recursion limit in step (g) -- this cap exists so
a judge that never approves doesn't loop forever; the recursion limit is a
separate, LangGraph-enforced safety net regardless of your own logic.

### Implementation
Same `START -> write_draft -> judge_draft -> (loop | END)` shape as step
(c)'s counter loop, but `write_draft` and `judge_draft` are both real LLM
calls, and the loop condition is the judge's verdict, not a counter.

In [ ]:
class RefineState(TypedDict):
    topic: str
    draft: str
    feedback: str
    approved: bool
    attempts: int


def write_draft(state: RefineState) -> RefineState:
    prompt = f"Write a two-line haiku about: {state['topic']}."
    if state["feedback"]:
        prompt += f" Address this feedback from a previous attempt: {state['feedback']}"
    draft = get_text(get_llm().invoke(prompt))
    return {"draft": draft, "attempts": state["attempts"] + 1}


def judge_draft(state: RefineState) -> RefineState:
    verdict = get_text(
        get_llm().invoke(
            f"Does this haiku attempt clearly relate to '{state['topic']}' and read like a haiku? "
            f"Reply with exactly 'APPROVED' if yes, otherwise one short sentence of feedback.\n\n{state['draft']}"
        )
    ).strip()
    approved = verdict.upper().startswith("APPROVED")
    return {"approved": approved, "feedback": "" if approved else verdict}


def should_refine(state: RefineState) -> Literal["write_draft", "__end__"]:
    if state["approved"] or state["attempts"] >= 3:
        return "__end__"
    return "write_draft"


refine_builder = StateGraph(RefineState)
refine_builder.add_node("write_draft", write_draft)
refine_builder.add_node("judge_draft", judge_draft)
refine_builder.add_edge(START, "write_draft")
refine_builder.add_edge("write_draft", "judge_draft")
refine_builder.add_conditional_edges("judge_draft", should_refine, {"write_draft": "write_draft", "__end__": END})
refine_app = refine_builder.compile()

result = refine_app.invoke(
    {"topic": "debugging code at midnight", "draft": "", "feedback": "", "approved": False, "attempts": 0}
)
print(f"Attempts taken: {result['attempts']}  |  Approved: {result['approved']}")
print(result["draft"])
print()
print(refine_app.get_graph().draw_mermaid())

### Mermaid diagram — Step (c variant) topology

```mermaid
graph TD;
    START([START]) --> write_draft["write_draft (LLM call)"];
    write_draft --> judge_draft["judge_draft (LLM call)"];
    judge_draft -- not approved, attempts < 3 --> write_draft;
    judge_draft -- approved or attempts >= 3 --> END([END]);
```

## Step (d) — Checkpointing / Memory

### Theory
By default, a compiled graph's state exists only for the duration of one
`.invoke()` call. A **checkpointer** (e.g. `InMemorySaver`) persists state
after every step, keyed by a `thread_id`. Passing the same `thread_id` on a
later `.invoke()` call resumes from where that thread left off — this is
how multi-turn conversational memory works, and it's also the prerequisite
mechanism for human-in-the-loop (step i), since pausing/resuming requires
persisted state.

**Related concept:** checkpointing is *not* the same as a vector-store-backed
long-term memory (used in RAG) — it's short-term, thread-scoped state
persistence for the graph's own execution, not a retrieval system.

### Implementation
Same `increment` node from step (c), but compiled with an `InMemorySaver`
and invoked twice on the same `thread_id` — the counter carries over between
calls, proving state persisted.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

memory_builder = StateGraph(LoopState)
memory_builder.add_node("increment", increment)
memory_builder.add_edge(START, "increment")
memory_builder.add_edge("increment", END)

memory_app = memory_builder.compile(checkpointer=checkpointer)

thread = {"configurable": {"thread_id": "demo-thread-1"}}

r1 = memory_app.invoke({"counter": 0, "log": []}, thread)
print("Call 1 (fresh counter seed):", r1)

# Second call omits counter -- LangGraph pulls the persisted state for this
# thread_id and merges the new input on top of it.
r2 = memory_app.invoke({"log": []}, thread)
print("Call 2 (same thread_id, counter persisted):", r2)

print()
print("Checkpoint history length for this thread:", len(list(memory_app.get_state_history(thread))))

### Mermaid diagram — Step (d) topology

```mermaid
graph TD;
    START([START]) --> increment[increment];
    increment --> END([END]);
    note[("checkpointer persists\nstate per thread_id")]:::note
    classDef note fill:#fff3cd,stroke:#333,stroke-dasharray: 3 3
```

## Step (d variant) — Same Checkpointing Shape, Real Conversational Memory

### Theory
Step (d) proved checkpointing persists a plain counter across calls. This
variant is the actual use case checkpointing exists for: a `chat` node
that calls the LLM with the *entire message list* from state (via the
`add_messages` reducer, which appends rather than overwrites, same reducer
concept as `operator.add` in step c/l), compiled with the same
`InMemorySaver`. Invoking twice on the same `thread_id` means the second
call's LLM request includes the first turn's messages -- so the model can
genuinely reference something said earlier, not because we re-sent it
manually, but because the checkpointer restored it.

**Related concept:** this is the same `add_messages`/`ToolState` pattern
used for the tool-calling agent in step (e) -- that step's agent already
implicitly benefits from this if compiled with a checkpointer, though we
didn't add one there since step (e)'s focus was tool routing, not memory.

### Implementation
Same `START -> chat -> END` shape as step (d), compiled with the same
`InMemorySaver`, invoked twice on one `thread_id` -- second turn asks about
something only mentioned in the first turn.

In [ ]:
from langchain_core.messages import HumanMessage as _HumanMessage
from langgraph.graph.message import add_messages as _add_messages


class ChatMemoryState(TypedDict):
    messages: Annotated[list, _add_messages]


def chat_node(state: ChatMemoryState) -> ChatMemoryState:
    return {"messages": [get_llm().invoke(state["messages"])]}


chat_builder = StateGraph(ChatMemoryState)
chat_builder.add_node("chat", chat_node)
chat_builder.add_edge(START, "chat")
chat_builder.add_edge("chat", END)
chat_app = chat_builder.compile(checkpointer=InMemorySaver())

chat_thread = {"configurable": {"thread_id": "memory-demo-1"}}

r1 = chat_app.invoke({"messages": [_HumanMessage(content="My favorite programming language is Rust. Remember that.")]}, chat_thread)
print("Turn 1:", get_text(r1["messages"][-1]))

r2 = chat_app.invoke({"messages": [_HumanMessage(content="What's my favorite programming language?")]}, chat_thread)
print("Turn 2 (relies on memory from turn 1):", get_text(r2["messages"][-1]))
print()
print(chat_app.get_graph().draw_mermaid())

### Mermaid diagram — Step (d variant) topology (identical shape to step d)

```mermaid
graph TD;
    START([START]) --> chat["chat (LLM call)"];
    chat --> END([END]);
    note[("checkpointer persists\nfull message history per thread_id")]:::note
    classDef note fill:#fff3cd,stroke:#333,stroke-dasharray: 3 3
```

## Step (e) — Tool Calling

### Theory
Tool calling lets the LLM itself decide to invoke a Python function, based
on the function's name/description/schema (via `@tool` and `.bind_tools()`).
The LLM doesn't execute the tool — it returns a *request* (`tool_calls` on
the response), and your graph is responsible for actually running it (via a
`ToolNode`) and feeding the result back.

The standard LangGraph pattern: an `agent` node calls the LLM, then a
conditional edge (`tools_condition`) checks whether the response contains
tool calls — if yes, route to a `ToolNode`, run the tool, and loop back to
the agent with the tool's result; if no, the agent's answer is final.

**Related concept:** this conditional-edge-based routing is the same
mechanism from step (b) — `tools_condition` is just a pre-built routing
function LangGraph ships for exactly this pattern.

### Implementation
One `calculator` tool, bound to the LLM, wired as `agent <-> tools` with
`tools_condition` deciding whether to loop into the tool or finish.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '12 * (4 + 1)'."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: expression contains disallowed characters."
    return str(eval(expression, {"__builtins__": {}}))


class ToolState(TypedDict):
    messages: Annotated[list, add_messages]


calc_llm = get_llm().bind_tools([calculator])


def agent_node(state: ToolState) -> ToolState:
    return {"messages": [calc_llm.invoke(state["messages"])]}


tool_builder = StateGraph(ToolState)
tool_builder.add_node("agent", agent_node)
tool_builder.add_node("tools", ToolNode([calculator]))
tool_builder.add_edge(START, "agent")
tool_builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
tool_builder.add_edge("tools", "agent")

tool_app = tool_builder.compile()

out = tool_app.invoke({"messages": [HumanMessage(content="What is 234 * 17? Use the calculator tool.")]})
for m in out["messages"]:
    print(type(m).__name__, "-", getattr(m, "content", "") or getattr(m, "tool_calls", ""))
print()
print(tool_app.get_graph().draw_mermaid())

### Mermaid diagram — Step (e) topology

```mermaid
graph TD;
    START([START]) --> agent[agent];
    agent -- has tool_calls --> tools[tools];
    agent -- no tool_calls --> END([END]);
    tools --> agent;
```

## Step (f) — Multi-Tool Selection: How Does the LLM Decide Which Tool to Call?

### Theory
When multiple tools are bound (`bind_tools([t1, t2, ...])`), each tool's
name, docstring, and parameter schema are sent to the LLM as part of the
request (as JSON schemas, not just names). The LLM's tool-selection
decision is **not** hardcoded logic in LangGraph — it's the LLM reading the
user's message plus every tool's description and choosing (via its own
training on structured tool-use) which single schema best matches the
intent, then filling in the arguments according to that schema.

This means **tool descriptions matter enormously** — a vague docstring
produces wrong tool selection; a precise, example-rich one produces
reliable routing. This is the real lever you control, not a routing
`if/else` you write yourself.

**Related concept:** this differs from step (b)'s conditional edges, which
route based on code you write inspecting state. Here, the LLM itself is the
router — LangGraph just executes whichever tool call comes back.

### Implementation
Six free tools (calculator, toy weather, DuckDuckGo search, Wikipedia,
Python REPL, arXiv search) bound together. Several different questions are
sent through and we print which tool the LLM picked for each — proving
selection is driven by matching intent to tool description, not by us.

The first loop below inspects selection only. The second graph runs the
selected tool through `ToolNode` and sends the `ToolMessage` back to the
model, so students can see the complete message trajectory.

In [ ]:
from ddgs import DDGS
import wikipedia
import arxiv as arxiv_lib

TOY_WEATHER = {"paris": "18C, cloudy", "delhi": "34C, sunny", "tokyo": "22C, rainy"}


@tool
def get_weather(city: str) -> str:
    """Look up today's toy weather for a known city (paris, delhi, tokyo)."""
    return TOY_WEATHER.get(city.lower(), f"No toy weather data for '{city}'.")


@tool
def web_search(query: str) -> str:
    """Search the live web for current events, news, or anything not in your training data."""
    results = DDGS().text(query, max_results=3)
    return "\n".join(f"- {r['title']}: {r['href']}" for r in results) or "No results."


@tool
def wikipedia_lookup(topic: str) -> str:
    """Look up an encyclopedic summary of a well-known topic, concept, or person on Wikipedia."""
    hits = wikipedia.search(topic)
    if not hits:
        return f"No Wikipedia page found for '{topic}'."
    return wikipedia.summary(hits[0], sentences=2, auto_suggest=False)


@tool
# Teaching warning: restricted builtins are not a secure sandbox. Never run
# untrusted model-generated Python in the application process.
def python_repl(code_str: str) -> str:
    """Execute a short snippet of Python for numeric/data computations (e.g. statistics) and return the result. Assign the final answer to a variable named `result`."""
    scope: dict = {}
    try:
        exec(code_str, {"__builtins__": {}}, scope)
        return str(scope.get("result", "no `result` variable was set"))
    except Exception as exc:
        return f"Error: {exc}"


@tool
def arxiv_search(query: str) -> str:
    """Search arXiv for recent academic papers on a research topic."""
    client = arxiv_lib.Client()
    search = arxiv_lib.Search(query=query, max_results=2)
    return "\n".join(f"- {r.title}" for r in client.results(search)) or "No papers found."


all_tools = [calculator, get_weather, web_search, wikipedia_lookup, python_repl, arxiv_search]
multi_llm = get_llm().bind_tools(all_tools)

questions = [
    "What's the weather in Delhi?",
    "What is 987 divided by 3?",
    "Who founded the company OpenAI? Look it up on Wikipedia.",
    "Find one recent arXiv paper about graph neural networks.",
    "Compute the standard deviation of [4, 8, 15, 16, 23, 42] using Python.",
    "Search the web for the latest LangGraph release version.",
]

for q in questions:
    resp = multi_llm.invoke(q)
    chosen = [tc["name"] for tc in resp.tool_calls] if resp.tool_calls else "NONE (answered directly)"
    print(f"Q: {q}\n  -> tool selected: {chosen}\n")

multi_tool_builder = StateGraph(ToolState)
multi_tool_builder.add_node("agent", lambda s: {"messages": [multi_llm.invoke(s["messages"])]})
multi_tool_builder.add_node("tools", ToolNode(all_tools))
multi_tool_builder.add_edge(START, "agent")
multi_tool_builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
multi_tool_builder.add_edge("tools", "agent")
multi_tool_app = multi_tool_builder.compile()

multi_tool_result = multi_tool_app.invoke({"messages": [HumanMessage(content="What is the weather in Delhi? Use the weather tool.")]})
print("\nComplete multi-tool trajectory:")
for message in multi_tool_result["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        print(type(message).__name__, "tool_calls=", [call["name"] for call in message.tool_calls])
    else:
        print(type(message).__name__, "content=", get_text(message)[:180])

message_types = [type(message).__name__ for message in multi_tool_result["messages"]]
assert message_types[0] == "HumanMessage"
assert "ToolMessage" in message_types
assert message_types[-1] == "AIMessage"
print("Message sequence assertion passed:", message_types)

### Mermaid diagram — Step (f) topology (same shape as step e, more tool options)

```mermaid
graph TD;
    START([START]) --> agent[agent];
    agent -- picks one of 6 tools --> tools[tools: calculator / weather / web_search / wikipedia / python_repl / arxiv];
    agent -- no tool needed --> END([END]);
    tools --> agent;
```

## Step (g) — Recursion Limit

### Theory
LangGraph caps the number of **super-steps** a single `.invoke()` can take
(default 25) via the `recursion_limit` config, as a safety net independent
of your own loop-termination logic (step c). If a graph genuinely needs
more steps than the limit, or a termination condition has a bug and never
becomes true, LangGraph raises `GraphRecursionError` rather than looping
forever.

**Related concept:** don't confuse this with Python's own recursion limit
(`sys.setrecursionlimit`) — this is LangGraph's graph-execution step
counter, unrelated to the Python call stack.

### Implementation
Reuse step (c)'s loop, but deliberately set `recursion_limit` lower than the
number of passes the loop needs, and catch the resulting error — this is
the *expected* outcome of this cell, not a bug.

### Practice — make the tool loop yours

1. Add a second tool call to the prompt and inspect whether the model emits one or multiple tool calls.
2. Replace the weather question with a calculator question and compare the message sequence.
3. Add an assertion that the expected tool name appears in the AI message's `tool_calls`.

What would happen if a tool returned an error string? Would the graph stop, retry, or ask the model to recover?

In [ ]:
from langgraph.errors import GraphRecursionError


class RunawayState(TypedDict):
    counter: int


def bump(state: RunawayState) -> RunawayState:
    return {"counter": state["counter"] + 1}


def keep_going(state: RunawayState) -> Literal["bump", "__end__"]:
    # Deliberately requires far more passes than the recursion_limit below.
    return "bump" if state["counter"] < 50 else "__end__"


runaway_builder = StateGraph(RunawayState)
runaway_builder.add_node("bump", bump)
runaway_builder.add_edge(START, "bump")
runaway_builder.add_conditional_edges("bump", keep_going, {"bump": "bump", "__end__": END})
runaway_app = runaway_builder.compile()

try:
    runaway_app.invoke({"counter": 0}, {"recursion_limit": 5})
    print("Did not raise -- unexpected.")
except GraphRecursionError as exc:
    print(f"GraphRecursionError raised as expected: {exc}")

### Mermaid diagram — Step (g) topology (same as step c; the limit is a config, not a graph shape)

```mermaid
graph TD;
    START([START]) --> bump[bump];
    bump -- counter < 50 --> bump;
    bump -- counter >= 50 --> END([END]);
    limit["recursion_limit=5\nstops execution early"]:::note
    classDef note fill:#ffe0e0,stroke:#333,stroke-dasharray: 3 3
```

## Step (h) — Streaming

### Theory
`.invoke()` blocks until the whole graph finishes. `.stream()` instead
yields incrementally as execution progresses, which matters for any
live-facing demo/UI. Key `stream_mode` values:

- `"values"` — yields the full state after each super-step.
- `"updates"` — yields just the delta each node returned.
- `"messages"` — yields LLM token-level output as it's generated (for
  chat-style nodes).

**Related concept:** streaming is orthogonal to checkpointing (step d) and
tool calling (step e/f) — you can stream a graph that also has a
checkpointer and tools; they compose.

### Implementation
Re-run step (e)'s tool-calling graph via `.stream(..., stream_mode="updates")`
and print each incremental update as it arrives.

In [ ]:
print("Streaming step (e)'s tool-calling graph with stream_mode='updates':\n")
for chunk in tool_app.stream(
    {"messages": [HumanMessage(content="What is 55 + 45? Use the calculator.")]},
    stream_mode="updates",
):
    for node_name, node_output in chunk.items():
        print(f"[{node_name}] ->", node_output)

### Mermaid diagram — Step (h) topology (identical to step e; streaming changes how output is consumed, not the graph shape)

```mermaid
graph TD;
    START([START]) --> agent[agent];
    agent -- has tool_calls --> tools[tools];
    agent -- no tool_calls --> END([END]);
    tools --> agent;
```

## Step (i) — Human-in-the-Loop

### Theory
`interrupt(payload)` pauses graph execution mid-node and surfaces `payload`
to the caller — the graph is genuinely suspended (thanks to the
checkpointer from step d), not just blocked in a loop. Execution resumes
later via `.invoke(Command(resume=<value>), thread_config)`, and the
`interrupt()` call returns that resume value inside the node, continuing
from exactly where it paused.

This is the real mechanism behind "pause for human approval before this
step" workflows — the payload is whatever context the human needs to
decide, and `Command(resume=...)` carries their decision back in.

**Related concept:** interrupts require a checkpointer (step d) to work at
all — without persisted state, there's nothing to resume from.

### Implementation
A 3-node graph: `propose` sets a value, `approval` calls `interrupt()` to
pause for a human decision, `finalize` applies it. We invoke once (pauses),
inspect the pending interrupt, then resume with an approval decision.

In [ ]:
from langgraph.types import interrupt, Command


class ApprovalState(TypedDict):
    value: int
    approved: bool


def propose(state: ApprovalState) -> ApprovalState:
    return {"value": 100}


def approval(state: ApprovalState) -> ApprovalState:
    decision = interrupt({"question": f"Approve applying value={state['value']}? (True/False)"})
    return {"approved": decision}


def finalize(state: ApprovalState) -> ApprovalState:
    return {"value": state["value"] if state["approved"] else 0}


hitl_builder = StateGraph(ApprovalState)
hitl_builder.add_node("propose", propose)
hitl_builder.add_node("approval", approval)
hitl_builder.add_node("finalize", finalize)
hitl_builder.add_edge(START, "propose")
hitl_builder.add_edge("propose", "approval")
hitl_builder.add_edge("approval", "finalize")
hitl_builder.add_edge("finalize", END)

hitl_app = hitl_builder.compile(checkpointer=InMemorySaver())
hitl_thread = {"configurable": {"thread_id": "approval-demo-1"}}

paused = hitl_app.invoke({"value": 0, "approved": False}, hitl_thread)
print("After first invoke, graph is paused. Pending interrupt:")
print(paused.get("__interrupt__"))
print("Node waiting to run next:", hitl_app.get_state(hitl_thread).next)

# Simulate a human approving the action.
resumed = hitl_app.invoke(Command(resume=True), hitl_thread)
print("\nAfter resuming with approval=True:", resumed)

### Mermaid diagram — Step (i) topology

```mermaid
graph TD;
    START([START]) --> propose[propose];
    propose --> approval["approval (interrupt: pauses for human)"];
    approval --> finalize[finalize];
    finalize --> END([END]);
```

## Step (j) — Subgraphs

### Theory
A **compiled graph is itself a valid node** in another graph — this is how
LangGraph composes larger systems from smaller, independently-testable
pieces. The parent graph's state and the subgraph's state can be the same
schema (shared keys pass through directly) or different (requiring a small
transform function at the boundary). Here we use a shared-schema subgraph,
the simplest case.

**Related concept:** a subgraph is not the same as the orchestrator-worker
pattern (step l) — a subgraph is static composition (always runs, wired at
build time); orchestrator-worker dynamically decides *how many* workers to
spawn at runtime via `Send`.

### Implementation
A small 2-node subgraph (`double` -> `stringify`) compiled on its own, then
plugged directly into a parent graph as a single node.

In [ ]:
class SubState(TypedDict):
    number: int
    label: str


def double(state: SubState) -> SubState:
    return {"number": state["number"] * 2}


def stringify(state: SubState) -> SubState:
    return {"label": f"result={state['number']}"}


sub_builder = StateGraph(SubState)
sub_builder.add_node("double", double)
sub_builder.add_node("stringify", stringify)
sub_builder.add_edge(START, "double")
sub_builder.add_edge("double", "stringify")
sub_builder.add_edge("stringify", END)
sub_app = sub_builder.compile()  # a fully independent, testable graph

# Plug the compiled subgraph directly into a parent graph as one node.
parent_builder = StateGraph(SubState)
parent_builder.add_node("sub_process", sub_app)
parent_builder.add_edge(START, "sub_process")
parent_builder.add_edge("sub_process", END)
parent_app = parent_builder.compile()

print("Subgraph alone:", sub_app.invoke({"number": 5, "label": ""}))
print("Parent graph (wrapping the subgraph):", parent_app.invoke({"number": 5, "label": ""}))
print()
print(parent_app.get_graph().draw_mermaid())

### Mermaid diagram — Step (j) topology

```mermaid
graph TD;
    subgraph parent_app
    START([START]) --> sub_process["sub_process (subgraph)"];
    sub_process --> END([END]);
    end
    subgraph sub_process_internal["sub_process internals"]
    sd_start([START]) --> double[double] --> stringify[stringify] --> sd_end([END]);
    end
```

## Step (k) — Sequential vs Parallel Execution

### Theory
LangGraph executes in **super-steps**: all nodes scheduled for the current
super-step run, then the graph advances. A **sequential** chain (`A -> B ->
C`) is one node per super-step — three super-steps total. A **parallel
fan-out** (`START -> A, START -> B, START -> C` all as separate edges) puts
`A`, `B`, `C` in the *same* super-step, so LangGraph runs them concurrently,
then a fan-in node (`A/B/C -> join`) waits for all three before proceeding.

This is a real wall-clock difference for I/O-bound nodes (e.g. each calling
an LLM or an API) — sequential pays each node's latency serially; parallel
pays roughly the slowest single node's latency.

**Related concept:** parallel fan-out here is *static* (always exactly 3
branches, wired at build time) — contrast with step (l)'s `Send`-based
fan-out, which is *dynamic* (however many workers the orchestrator decides
on at runtime).

### Implementation
Three identical "slow" nodes (simulated with `time.sleep`), wired once
sequentially and once in parallel, with wall-clock time compared.

In [ ]:
class TimingState(TypedDict):
    results: Annotated[list[str], operator.add]


def slow_task(name):
    def _node(state: TimingState) -> TimingState:
        time.sleep(0.5)
        return {"results": [name]}
    return _node


# --- Sequential: A -> B -> C ---
seq_builder = StateGraph(TimingState)
seq_builder.add_node("task_a", slow_task("a"))
seq_builder.add_node("task_b", slow_task("b"))
seq_builder.add_node("task_c", slow_task("c"))
seq_builder.add_edge(START, "task_a")
seq_builder.add_edge("task_a", "task_b")
seq_builder.add_edge("task_b", "task_c")
seq_builder.add_edge("task_c", END)
seq_app = seq_builder.compile()

t0 = time.time()
seq_result = seq_app.invoke({"results": []})
seq_time = time.time() - t0

# --- Parallel: START fans out to all three, join waits for all ---
par_builder = StateGraph(TimingState)
par_builder.add_node("task_a", slow_task("a"))
par_builder.add_node("task_b", slow_task("b"))
par_builder.add_node("task_c", slow_task("c"))
par_builder.add_node("join", lambda state: {})
par_builder.add_edge(START, "task_a")
par_builder.add_edge(START, "task_b")
par_builder.add_edge(START, "task_c")
par_builder.add_edge("task_a", "join")
par_builder.add_edge("task_b", "join")
par_builder.add_edge("task_c", "join")
par_builder.add_edge("join", END)
par_app = par_builder.compile()

t0 = time.time()
par_result = par_app.invoke({"results": []})
par_time = time.time() - t0

print(f"Sequential: {seq_result}  took {seq_time:.2f}s")
print(f"Parallel:   {par_result}  took {par_time:.2f}s")

### Mermaid diagrams — Step (k) topologies

Sequential:
```mermaid
graph TD;
    START([START]) --> task_a --> task_b --> task_c --> END([END]);
```

Parallel:
```mermaid
graph TD;
    START([START]) --> task_a;
    START --> task_b;
    START --> task_c;
    task_a --> join;
    task_b --> join;
    task_c --> join;
    join --> END([END]);
```

## Step (l) — Orchestrator-Worker Pattern

### Theory
The orchestrator-worker pattern: one **orchestrator** node (usually an LLM
call) decides how to break a task into an arbitrary number of subtasks,
then dynamically dispatches one `Send(node_name, payload)` per subtask —
this is the *dynamic* fan-out mentioned in step (k), where the branch count
is decided at runtime, not fixed at graph-build time. A **worker** node
processes one subtask each (all in the same super-step, run concurrently),
and their results accumulate into a shared state field via a **reducer**
(`Annotated[list, operator.add]`, first seen back in step (c)'s `log`
field) — each worker's returned list gets appended, not overwritten.

**Related concept:** this is the general form of step (k)'s parallel
fan-out — same underlying concurrent-superstep mechanism, but the *number*
of parallel branches is a runtime decision instead of a fixed graph shape.

### Implementation
An orchestrator LLM call splits a broad topic into 3 subtopics; `Send` fans
those out to worker nodes that each write one summary line; a synthesizer
node combines all worker outputs into a final answer.

In [ ]:
from langgraph.types import Send


class OrchestratorState(TypedDict):
    topic: str
    subtopics: list[str]
    sections: Annotated[list[str], operator.add]
    final_report: str


class WorkerState(TypedDict):
    subtopic: str
    sections: Annotated[list[str], operator.add]


def orchestrator_node(state: OrchestratorState) -> OrchestratorState:
    prompt = (
        f"Break the topic '{state['topic']}' into exactly 3 short subtopics. "
        "Reply with just the 3 subtopics, one per line, no numbering."
    )
    response = get_text(get_llm().invoke(prompt))
    subtopics = [line.strip("- ").strip() for line in response.strip().split("\n") if line.strip()][:3]
    return {"subtopics": subtopics}


def assign_workers(state: OrchestratorState):
    return [Send("worker", {"subtopic": s, "sections": []}) for s in state["subtopics"]]


def worker_node(state: WorkerState) -> OrchestratorState:
    summary = get_text(get_llm().invoke(f"Write one sentence about: {state['subtopic']}"))
    return {"sections": [f"[{state['subtopic']}] {summary}"]}


def synthesizer_node(state: OrchestratorState) -> OrchestratorState:
    return {"final_report": "\n".join(state["sections"])}


orch_builder = StateGraph(OrchestratorState)
orch_builder.add_node("orchestrator", orchestrator_node)
orch_builder.add_node("worker", worker_node)
orch_builder.add_node("synthesizer", synthesizer_node)
orch_builder.add_edge(START, "orchestrator")
orch_builder.add_conditional_edges("orchestrator", assign_workers, ["worker"])
orch_builder.add_edge("worker", "synthesizer")
orch_builder.add_edge("synthesizer", END)

orch_app = orch_builder.compile()

result = orch_app.invoke({"topic": "LangGraph", "subtopics": [], "sections": [], "final_report": ""})
print("Subtopics chosen by orchestrator:", result["subtopics"])
print()
print("Final report:\n" + result["final_report"])

### Mermaid diagram — Step (l) topology

```mermaid
graph TD;
    START([START]) --> orchestrator[orchestrator];
    orchestrator -- Send: dynamic N workers --> worker[worker];
    worker --> synthesizer[synthesizer];
    synthesizer --> END([END]);
```

## Recap

| Step | Concept | Key API |
|---|---|---|
| a | StateGraph basics (+ LLM-as-node variant) | `StateGraph`, `add_node`, `add_edge`, `START`/`END` |
| b | Conditional edges (+ LLM-as-router variant) | `add_conditional_edges` |
| c | Loop / cycle (+ LLM self-refinement variant) | conditional edge routing back to itself |
| d | Checkpointing / memory (+ real chat-memory variant) | `InMemorySaver`, `thread_id` |
| e | Tool calling | `bind_tools`, `ToolNode`, `tools_condition` |
| f | Multi-tool selection | LLM matches intent to tool schema/description |
| g | Recursion limit | `GraphRecursionError`, `recursion_limit` config |
| h | Streaming | `.stream(stream_mode=...)` |
| i | Human-in-the-loop | `interrupt()`, `Command(resume=...)` |
| j | Subgraphs | a compiled graph used as a node |
| k | Sequential vs parallel | super-steps; static fan-out/fan-in |
| l | Orchestrator-worker | `Send`, dynamic fan-out, reducers |

### Practice checklist

- Add a third conditional branch in step (b).
- Change a reducer from overwrite behavior to append behavior.
- Add a retry cap to the self-refinement loop.
- Resume the approval graph with approve, reject, and escalation inputs.
- Create a new checkpoint thread and compare its state with the original.
- Convert the sequential graph in step (k) into fan-out/fan-in execution.

This progression is the standard prerequisite path before building agentic
RAG in LangGraph — retrieval-as-a-tool, self-grading loops, and multi-source
routing (discussed earlier in this conversation) are direct applications of
steps (c), (e)/(f), and (l) above.